In [ ]:
import requests
import json

import numpy as np
import pandas as pd
import geopandas as gpd

from scipy.stats import pearsonr

import folium
import mapclassify
import contextily as ctx
import branca.colormap as cm

import matplotlib.pyplot as plt
import seaborn as sns

# **Demographics & Raster Values**

- https://www.census.gov/content/dam/Census/library/publications/2018/acs/acs_general_handbook_2018_ch08.pdf

## **Raster Values**

In [ ]:
raster_values = gpd.read_file('data/tracts_with_raster_values.shp')
raster_values = raster_values.to_crs('EPSG:3857')
raster_values['GEOID'] = '36047' + raster_values['CT2010']
raster_values = raster_values.drop(columns=['CT2010'])

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(35,35))

raster_values.plot(column='lst', cmap='Reds', figsize=(30,30), legend=True, missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, legend_kwds={'shrink': 0.75}, ax=ax)
ctx.add_basemap(ax, zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);
ax.set_title('Land Surface Temperature in Celsius', fontsize=40, fontweight='bold')
ax.set_axis_off();

plt.savefig('images/lst.png', facecolor='white', transparent=False, dpi=150)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(35,35))

raster_values.plot(column='ndvi', cmap='Greens', figsize=(30,30), legend=True, missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, legend_kwds={'shrink': 0.75}, ax=ax)
ctx.add_basemap(ax, zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);
ax.set_title('Normalized Difference Vegetation Index (NDVI)', fontsize=40, fontweight='bold')
ax.set_axis_off();

plt.savefig('images/ndvi.png', dpi=150)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(35,35))

color_map = plt.cm.get_cmap('Greys')
reversed_color_map = color_map.reversed()

raster_values.plot(column='albedo', cmap=reversed_color_map, figsize=(30,30), legend=True, missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, legend_kwds={'shrink': 0.75}, ax=ax)
ctx.add_basemap(ax, zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);
ax.set_title('Albedo', fontsize=40, fontweight='bold')
ax.set_axis_off();

plt.savefig('images/albedo.png', dpi=150)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(50,10))
ax = ax.flatten()

raster_values.plot(column='lst', cmap='Reds', legend=True, missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, ax=ax[0])
ctx.add_basemap(ax[0], zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);
ax[0].set_title('Land Surface Temperature in Celsius')
ax[0].set_axis_off();

raster_values.plot(column='ndvi', cmap='Greens', legend=True, missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, ax=ax[1])
ctx.add_basemap(ax[1], zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);
ax[1].set_title('Normalized Difference Vegetation Index (NDVI)')
ax[1].set_axis_off();

color_map = plt.cm.get_cmap('Greys')
reversed_color_map = color_map.reversed()
raster_values.plot(column='albedo', cmap=reversed_color_map, legend=True, missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, ax=ax[2])
ctx.add_basemap(ax[2], zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);
ax[2].set_title('Albedo')
ax[2].set_axis_off();

plt.tight_layout();

plt.savefig('images/raster_values.png', dpi=300)

## **Demographics**

In [ ]:
base_url = f'https://api.census.gov/data/2019/acs/acs5'
data_url = f'?get=B17013_001E,B17013_001M,B17013_002E,B17013_002M'
key_url = f'&key=53e74525302268cad5f2d45b2cfc8da890a7d2d8'    
geography_url = f'&for=tract:*&in=county:047&in=state:36'

url = base_url + data_url + geography_url + key_url

json = requests.get(url).json()
df_family_poverty = pd.DataFrame(json[1:], columns=json[0])

table_names = ['number_families_e',
               'number_families_moe',
               'number_families_below_poverty_line_e',
               'number_families_below_poverty_line_moe']
table_names.append('State')
table_names.append('County')
table_names.append('Tract')
df_family_poverty.columns = table_names

df_family_poverty['GEOID'] = df_family_poverty['State'] + df_family_poverty['County'] + df_family_poverty['Tract']
df_family_poverty = df_family_poverty.drop(columns=['State', 'County', 'Tract'])

tracts = gpd.read_file('https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/NYC_Census_Tracts_for_2010_US_Census/FeatureServer/0/query?where=1=1&outFields=*&outSR=4326&f=pgeojson')
tracts = tracts[tracts['BoroName'] == 'Brooklyn']
tracts['GEOID'] = '36047' + tracts['CT2010']
tracts = gpd.GeoDataFrame(tracts[['GEOID', 'geometry']])
tracts = tracts.to_crs('EPSG:26918')

df_family_poverty = df_family_poverty.merge(tracts, on=['GEOID'], how='outer')
df_family_poverty = gpd.GeoDataFrame(df_family_poverty)

for column in df_family_poverty.columns.tolist()[:-2]:
    df_family_poverty[column] = df_family_poverty[column].astype(float)
    
df_family_poverty['proportion_families_below_poverty_line_e'] = df_family_poverty['number_families_below_poverty_line_e']/df_family_poverty['number_families_e']
df_family_poverty['proportion_families_below_poverty_line_moe'] = (1/df_family_poverty['number_families_e']) * np.sqrt((df_family_poverty['number_families_below_poverty_line_moe'])**2 - ((df_family_poverty['proportion_families_below_poverty_line_e'])**2 * (df_family_poverty['number_families_moe'])**2))

df_family_poverty['proportion_families_below_poverty_line_e'] = df_family_poverty['proportion_families_below_poverty_line_e'] * 100
df_family_poverty['proportion_families_below_poverty_line_moe'] = df_family_poverty['proportion_families_below_poverty_line_moe'] * 100
df_family_poverty = gpd.GeoDataFrame(df_family_poverty[['GEOID', 'proportion_families_below_poverty_line_e', 'proportion_families_below_poverty_line_moe', 'geometry']])

df_family_poverty['proportion_families_below_poverty_line_se'] = df_family_poverty['proportion_families_below_poverty_line_moe'] / 1.645
df_family_poverty['proportion_families_below_poverty_line_cv'] = (df_family_poverty['proportion_families_below_poverty_line_se'] / df_family_poverty['proportion_families_below_poverty_line_e']) * 100

In [ ]:
df_family_poverty = df_family_poverty.to_crs('EPSG:3857')

fig, ax = plt.subplots(1, 1, figsize=(35,35))

df_family_poverty.plot(column='proportion_families_below_poverty_line_e', scheme='FisherJenks', k=10, legend=True, cmap='viridis', ax=ax, edgecolor='white', missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, legend_kwds={'loc':'lower right', 'fmt':'{:1.0f}%', 'fontsize':35, 'frameon':True})
df_family_poverty[df_family_poverty['proportion_families_below_poverty_line_cv'] > 40].plot(column='proportion_families_below_poverty_line_e', edgecolor='white', hatch = '/////', ax=ax)
ctx.add_basemap(ax, zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);

ax.set_title('Percentage of Families in Poverty', size=40, weight='bold')
ax.set_axis_off();

plt.savefig('images/poverty.png', facecolor='white', transparent=False, dpi=150)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(50,35))
ax = ax.flatten()

df_family_poverty.plot(column='proportion_families_below_poverty_line_e', scheme='FisherJenks', k=10, legend=True, cmap='viridis', ax=ax[0], edgecolor='white', missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, legend_kwds={'loc':'lower right', 'fmt':'{:1.0f}%', 'fontsize':35, 'frameon':True})
df_family_poverty[df_family_poverty['proportion_families_below_poverty_line_cv'] > 40].plot(column='proportion_families_below_poverty_line_e', edgecolor='white', hatch = '/////', ax=ax[0])
ctx.add_basemap(ax[0], zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);
ax[0].set_title('Percentage of Families in Poverty', size=40, weight='bold')
ax[0].set_axis_off();

raster_values.plot(column='lst', cmap='Reds', legend=False, missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, ax=ax[1])
ctx.add_basemap(ax[1], zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);
ax[1].set_title('Land Surface Temperature in Celsius', size=40, weight='bold')
ax[1].set_axis_off();

plt.tight_layout();
plt.savefig('images/poverty_lst.png', facecolor='white', transparent=False, dpi=100)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(50,35))
ax = ax.flatten()

df_family_poverty.plot(column='proportion_families_below_poverty_line_e', scheme='FisherJenks', k=10, legend=True, cmap='viridis', ax=ax[0], edgecolor='white', missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, legend_kwds={'loc':'lower right', 'fmt':'{:1.0f}%', 'fontsize':35, 'frameon':True})
df_family_poverty[df_family_poverty['proportion_families_below_poverty_line_cv'] > 40].plot(column='proportion_families_below_poverty_line_e', edgecolor='white', hatch = '/////', ax=ax[0])
ctx.add_basemap(ax[0], zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);
ax[0].set_title('Percentage of Families in Poverty', size=40, weight='bold')
ax[0].set_axis_off();

raster_values.plot(column='ndvi', cmap='Greens', legend=False, missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, ax=ax[1])
ctx.add_basemap(ax[1], zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);
ax[1].set_title('Normalized Difference Vegetation Index (NDVI)', size=40, weight='bold')
ax[1].set_axis_off();

plt.tight_layout();
plt.savefig('images/poverty_ndvi.png', dpi=300)

In [ ]:
base_url = f'https://api.census.gov/data/2019/acs/acs5'
data_url = f'?get=B03002_001E,B03002_001M,B03002_003E,B03002_003M,B03002_004E,B03002_004M,B03002_005E,B03002_005M,B03002_006E,B03002_006M,B03002_007E,B03002_007M,B03002_008E,B03002_008M,B03002_009E,B03002_009M,B03002_012E,B03002_012M'
key_url = f'&key=53e74525302268cad5f2d45b2cfc8da890a7d2d8'    
geography_url = f'&for=tract:*&in=county:047&in=state:36'

url = base_url + data_url + geography_url + key_url

json = requests.get(url).json()
df_race = pd.DataFrame(json[1:], columns=json[0])

table_names = ['population_e','population_m',
               'white_e','white_m',
               'black_e','black_m',
               'native_e','native_m',
               'asian_e','asian_m',
               'hawaii_e','hawaii_m',
               'other_e','other_m',
               'two_e','two_m',
               'latino_e','latino_m'
              ]
table_names.append('State')
table_names.append('County')
table_names.append('Tract')
df_race.columns = table_names

df_race['GEOID'] = df_race['State'] + df_race['County'] + df_race['Tract']
df_race = df_race.drop(columns=['State', 'County', 'Tract'])

tracts = gpd.read_file('https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/NYC_Census_Tracts_for_2010_US_Census/FeatureServer/0/query?where=1=1&outFields=*&outSR=4326&f=pgeojson')
tracts = tracts[tracts['BoroName'] == 'Brooklyn']
tracts['GEOID'] = '36047' + tracts['CT2010']
tracts = gpd.GeoDataFrame(tracts[['GEOID', 'geometry']])
tracts = tracts.to_crs('EPSG:26918')

df_race = df_race.merge(tracts, on=['GEOID'], how='outer')
df_race = gpd.GeoDataFrame(df_race)

for column in df_race.columns.tolist()[:-2]:
    df_race[column] = df_race[column].astype(float)
    
df_race['nonwhite_e'] = df_race['black_e'] + df_race['native_e'] + df_race['asian_e'] + df_race['hawaii_e'] + df_race['other_e'] + df_race['two_e'] + df_race['latino_e'] 
df_race['nonwhite_m'] = np.sqrt(df_race['black_m']**2 + df_race['native_m']**2 + df_race['asian_m']**2 + df_race['hawaii_m']**2 + df_race['other_m']**2 + df_race['two_m']**2 + df_race['latino_m']**2)

df_race['proportion_nonwhite_e'] = df_race['nonwhite_e']/df_race['population_e']
df_race['proportion_nonwhite_m'] = (1/df_race['population_e']) * np.sqrt((df_race['nonwhite_m'])**2 - ((df_race['proportion_nonwhite_e'])**2) * (df_race['population_m'])**2)

df_race['proportion_nonwhite_e'] = df_race['proportion_nonwhite_e'] * 100
df_race['proportion_nonwhite_m'] = df_race['proportion_nonwhite_m'] * 100

df_race = gpd.GeoDataFrame(df_race[['GEOID', 'proportion_nonwhite_e', 'proportion_nonwhite_m', 'geometry']])

df_race['proportion_nonwhite_se'] = df_race['proportion_nonwhite_m'] / 1.645
df_race['proportion_nonwhite_cv'] = (df_race['proportion_nonwhite_se'] / df_race['proportion_nonwhite_e']) * 100

In [ ]:
df_race = df_race.to_crs('EPSG:3857')


fig, ax = plt.subplots(1, 1, figsize=(35,35))

df_race.plot(column='proportion_nonwhite_e', scheme='FisherJenks', k=10, legend=True, cmap='viridis', ax=ax, edgecolor='white', missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, legend_kwds={'loc':'lower right', 'fmt':'{:1.0f}%', 'fontsize':35, 'frameon':True})
df_race[df_race['proportion_nonwhite_cv'] > 40].plot(column='proportion_nonwhite_e', edgecolor='white', hatch = '/////', ax=ax)
ctx.add_basemap(ax, zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);

ax.set_title('Percentage of Population Non-White', size=40, weight='bold')
ax.set_axis_off();

plt.savefig('images/nonwhite.png', dpi=150)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(50,35))
ax = ax.flatten()

df_race.plot(column='proportion_nonwhite_e', scheme='FisherJenks', k=10, legend=True, cmap='viridis', ax=ax[0], edgecolor='white', missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, legend_kwds={'loc':'lower right', 'fmt':'{:1.0f}%', 'fontsize':35, 'frameon':True})
df_race[df_race['proportion_nonwhite_cv'] > 40].plot(column='proportion_nonwhite_e', edgecolor='white', hatch = '/////', ax=ax[0])
ctx.add_basemap(ax[0], zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);
ax[0].set_title('Percentage of Population Non-White', size=40, weight='bold')
ax[0].set_axis_off();

raster_values.plot(column='lst', cmap='Reds', legend=False, missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, ax=ax[1])
ctx.add_basemap(ax[1], zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);
ax[1].set_title('Land Surface Temperature in Celsius', size=40, weight='bold')
ax[1].set_axis_off();

plt.tight_layout();
plt.savefig('images/nonwhite_lst.png', dpi=150)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(50,35))
ax = ax.flatten()

df_race.plot(column='proportion_nonwhite_e', scheme='FisherJenks', k=10, legend=True, cmap='viridis', ax=ax[0], edgecolor='white', missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, legend_kwds={'loc':'lower right', 'fmt':'{:1.0f}%', 'fontsize':35, 'frameon':True})
df_race[df_race['proportion_nonwhite_cv'] > 40].plot(column='proportion_nonwhite_e', edgecolor='white', hatch = '/////', ax=ax[0])
ctx.add_basemap(ax[0], zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);
ax[0].set_title('Percentage of Population Non-White', size=40, weight='bold')
ax[0].set_axis_off();

raster_values.plot(column='ndvi', cmap='Greens', legend=False, missing_kwds={"color": "lightgrey", "edgecolor": "white", "label": "No estimate"}, ax=ax[1])
ctx.add_basemap(ax[1], zoom=15,source=ctx.providers.CartoDB.PositronNoLabels);
ax[1].set_title('Normalized Difference Vegetation Index (NDVI)', size=40, weight='bold')
ax[1].set_axis_off();

plt.tight_layout();
plt.savefig('images/nonwhite_ndvi.png', dpi=300)

## **Correlation Matrix**

In [ ]:
raster_values_ = raster_values.drop(columns=['geometry'])
race = df_race.drop(columns=['geometry', 'proportion_nonwhite_m', 'proportion_nonwhite_se', 'proportion_nonwhite_cv'])
poverty = df_family_poverty.drop(columns=['geometry', 'proportion_families_below_poverty_line_moe', 'proportion_families_below_poverty_line_se', 'proportion_families_below_poverty_line_cv'])

df = raster_values_.merge(race, on=['GEOID']).merge(poverty, on=['GEOID'])
df.columns = ['LST', 'NDVI', 'Albedo', 'GEOID', 'Non-White', 'Poverty Rate']
df = df.drop(columns=['GEOID'])

In [ ]:
df.corr()

In [ ]:
df.corr(method=lambda x, y: pearsonr(x, y)[1]) - np.eye(len(df.columns)) 

In [ ]:
corr = df.corr()

fig, ax = plt.subplots(figsize=(30, 30))

mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.set(font_scale=1.8)
sns.heatmap(corr, annot=True, mask = mask, cmap=cmap,  annot_kws={
                'fontsize': 40,
                'fontweight': 'bold'
            });
plt.tick_params(axis='both', which='major', labelsize=40)
plt.savefig('images/correlation_matrix.png', dpi=200)

## **Interactive Map**

In [ ]:
df = gpd.GeoDataFrame(raster_values.drop(columns=['geometry']).merge(df_family_poverty.drop(columns=['geometry']), on=['GEOID']).merge(df_race, on=['GEOID']))

In [ ]:
_map = folium.Map(location=[40.6501, -73.94958], zoom_start=9, tiles=None)
folium.TileLayer("CartoDB positron", name="Carto Positron").add_to(_map)
folium.TileLayer('openstreetmap', name='OpenStreetMap').add_to(_map)
folium.TileLayer('Stamen Terrain', name='Stamen Terrain', control=False).add_to(_map)



choropleth_race = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Percent Population Non-White",
    data=df,
    columns=["GEOID", "proportion_nonwhite_e"],
    key_on="feature.properties.GEOID",
    fill_color="Purples",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Percent Population Non-White (2019)",
).add_to(_map)

choropleth_poverty = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Percent Families in Poverty",
    data=df,
    columns=["GEOID", "proportion_families_below_poverty_line_e"],
    key_on="feature.properties.GEOID",
    fill_color="Blues",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Percent Families in Poverty (2019)",
).add_to(_map)

choropleth_lst = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Land Surface Temperature (LST)",
    data=df,
    columns=["GEOID", "lst"],
    key_on="feature.properties.GEOID",
    fill_color="Reds",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Land Surface Temperature (LST)",
).add_to(_map)


choropleth_ndvi = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Normalized Difference Vegetation Index (NDVI)",
    data=df,
    columns=["GEOID", "ndvi"],
    key_on="feature.properties.GEOID",
    fill_color="Greens",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Normalized Difference Vegetation Index (NDVI)",
).add_to(_map)


choropleth_albedo = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Albedo",
    data=df,
    columns=["GEOID", "albedo"],
    key_on="feature.properties.GEOID",
    fill_color="Greys",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Albedo",
).add_to(_map)

style_function = lambda x: {'fillColor': '#ffffff', 
                            'color':'#000000', 
                            'fillOpacity': 0.1, 
                            'weight': 0.1}
highlight_function = lambda x: {'fillColor': '#000000', 
                                'color':'#000000', 
                                'fillOpacity': 0.50, 
                                'weight': 0.1}
tooltip = folium.features.GeoJson(
    df,
    style_function=style_function, 
    control=False,
    highlight_function=highlight_function, 
    tooltip=folium.features.GeoJsonTooltip(
        fields=['lst', 'ndvi', 'albedo', 'proportion_nonwhite_e', 'proportion_families_below_poverty_line_e'],
        aliases=['Land Surface Temperature', 'Normalized Difference Vegetation Index', 'Albedo', '% Population Non-White', '% Families in Poverty'],
        style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;") 
    )
)
_map.add_child(tooltip)
_map.keep_in_front(tooltip)


folium.LayerControl().add_to(_map)
_map

In [ ]:
_map.save('cgis_map1.html')

# **Building Morphology**

## **Morphometrics**

In [ ]:
buildings = gpd.read_file('data/brooklyn_buildings_with_raster_values.shp')
buildings = buildings.to_crs('EPSG:26918')
df = pd.read_csv('data/merged.csv')
df = df.merge(buildings, on='uID')
df = gpd.GeoDataFrame(df)

tracts = gpd.read_file('https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/NYC_Census_Tracts_for_2010_US_Census/FeatureServer/0/query?where=1=1&outFields=*&outSR=4326&f=pgeojson')
tracts = tracts.to_crs('EPSG:26918')
tracts = tracts[tracts.BoroName == 'Brooklyn']
tracts = tracts.drop(columns=['OBJECTID', 'CTLabel', 'BoroCode', 'BoroName', 'BoroCT2010', 'CDEligibil', 'NTACode', 'NTAName', 'PUMA', 'Shape__Area', 'Shape__Length'])
tracts = tracts.reset_index(drop=True)

df = gpd.sjoin(df, tracts, how='left')
df = df.drop(columns=['index_right', 'geometry', 'uID', 'eID', 'nID', 'bID', 'nodeID'])
df = df.groupby(['CT2010'], as_index=False).median()

df = gpd.GeoDataFrame(df.merge(tracts, on=['CT2010']))

In [ ]:
def plot_all_choropleths():
    
    fig, axes = plt.subplots(34, 2, figsize=(15,200))
    axes = axes.flatten()
    
    for index, column in enumerate(df.columns.tolist()[1:-1]):
        
        try:
            df.plot(column=column, scheme='FisherJenks', legend=True, ax=axes[index])
            axes[index].set_title(column)
            axes[index].set_axis_off();
        except:
            continue
            
plot_all_choropleths()

### **Streets**

In [ ]:
columns_streets = ['streetWidth', 'streetHeight', 'streetProfile', 'streetOpenness']

def plot_all_choropleths():
    
    fig, axes = plt.subplots(2, 2, figsize=(15,20))
    axes = axes.flatten()
    
    for index, column in enumerate(columns_streets):
        
        try:
            df.plot(column=column, scheme='FisherJenks', k=5, legend=True, ax=axes[index])
            axes[index].set_title(column)
            axes[index].set_axis_off();
        except:
            continue
            
plot_all_choropleths()

In [ ]:
_map = folium.Map(location=[40.6501, -73.94958], zoom_start=9, tiles=None)
folium.TileLayer("CartoDB positron", name="Carto Positron").add_to(_map)
folium.TileLayer('openstreetmap', name='OpenStreetMap').add_to(_map)
folium.TileLayer('Stamen Terrain', name='Stamen Terrain', control=False).add_to(_map)


myscale = (df['streetWidth'].quantile((0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1))).tolist()
choropleth_street_width = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Street Width",
    data=df,
    columns=["CT2010", "streetWidth"],
    key_on="feature.properties.CT2010",
    fill_color="YlGnBu",
    threshold_scale=myscale,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Street Width",
).add_to(_map)

myscale = (df['streetHeight'].quantile((0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1))).tolist()
choropleth_street_height = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Street Height",
    data=df,
    columns=["CT2010", "streetHeight"],
    key_on="feature.properties.CT2010",
    fill_color="YlGnBu",
    threshold_scale=myscale,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Street Height",
).add_to(_map)

myscale = (df['streetOpenness'].quantile((0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1))).tolist()
choropleth_street_openness = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Street Openness",
    data=df,
    columns=["CT2010", "streetOpenness"],
    key_on="feature.properties.CT2010",
    fill_color="YlGnBu",
    threshold_scale=myscale,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Street Openness",
).add_to(_map)

myscale = (df['streetProfile'].quantile((0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1))).tolist()
choropleth_street_profile = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Street Height/Width Ratio",
    data=df,
    columns=["CT2010", "streetProfile"],
    key_on="feature.properties.CT2010",
    fill_color="YlGnBu",
    threshold_scale=myscale,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Street Height/Width Ratio",
).add_to(_map)


style_function = lambda x: {'fillColor': '#ffffff', 
                            'color':'#000000', 
                            'fillOpacity': 0.1, 
                            'weight': 0.1}
highlight_function = lambda x: {'fillColor': '#000000', 
                                'color':'#000000', 
                                'fillOpacity': 0.50, 
                                'weight': 0.1}
tooltip = folium.features.GeoJson(
    df,
    style_function=style_function, 
    control=False,
    highlight_function=highlight_function, 
    tooltip=folium.features.GeoJsonTooltip(
        fields=['streetWidth', 'streetHeight', 'streetOpenness', 'streetProfile'],
        aliases=['Street Width', 'Street Height', 'Street Openness', 'Street Height/Width Ratio'],
        style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;") 
    )
)
_map.add_child(tooltip)
_map.keep_in_front(tooltip)


folium.LayerControl().add_to(_map)
_map

### **Buildings**

In [ ]:
columns_buildings = ['buildingHeight', 'buildingElongation', 'buildingFormFactor', 'buildingVolumeFacadeRatio',
                     'buildingSharedWallsRatio', 'buildingShapeIndex', 'buildingOrientation', 'buildingPerimeterWall', 'buildingMeanInterbuildingDistance',
                     'buildingBuildingAdjacency']

def plot_all_choropleths():
    
    fig, axes = plt.subplots(5, 2, figsize=(15,50))
    axes = axes.flatten()
    
    for index, column in enumerate(columns_buildings):
        
        try:
            df.plot(column=column, scheme='FisherJenks', k=5, legend=True, ax=axes[index])
            axes[index].set_title(column)
            axes[index].set_axis_off();
        except:
            continue
            
plot_all_choropleths()

In [ ]:
_map = folium.Map(location=[40.6501, -73.94958], zoom_start=9, tiles=None)
folium.TileLayer("CartoDB positron", name="Carto Positron").add_to(_map)
folium.TileLayer('openstreetmap', name='OpenStreetMap').add_to(_map)
folium.TileLayer('Stamen Terrain', name='Stamen Terrain', control=False).add_to(_map)


myscale = (df['buildingHeight'].quantile((0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1))).tolist()
choropleth_buildingHeight = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Building Height",
    data=df,
    columns=["CT2010", "buildingHeight"],
    key_on="feature.properties.CT2010",
    fill_color="YlGnBu",
    threshold_scale=myscale,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Building Height",
).add_to(_map)

myscale = (df['buildingElongation'].quantile((0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1))).tolist()
choropleth_buildingElongation = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Building Elongation",
    data=df,
    columns=["CT2010", "buildingElongation"],
    key_on="feature.properties.CT2010",
    fill_color="YlGnBu",
    threshold_scale=myscale,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Building Elongation",
).add_to(_map)

myscale = (df['buildingFormFactor'].quantile((0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1))).tolist()
choropleth_buildingFormFactor = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Building Form Factor",
    data=df,
    columns=["CT2010", "buildingFormFactor"],
    key_on="feature.properties.CT2010",
    fill_color="YlGnBu",
    threshold_scale=myscale,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Building Form Factor",
).add_to(_map)

myscale = (df['buildingVolumeFacadeRatio'].quantile((0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1))).tolist()
choropleth_buildingVolumeFacadeRatio = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Building Volume Facade Ratio",
    data=df,
    columns=["CT2010", "buildingVolumeFacadeRatio"],
    key_on="feature.properties.CT2010",
    fill_color="YlGnBu",
    threshold_scale=myscale,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Building Volume Facade Ratio",
).add_to(_map)

myscale = (df['buildingSharedWallsRatio'].quantile((0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1))).tolist()
choropleth_buildingSharedWallsRatio = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Building Shared Walls Ratio",
    data=df,
    columns=["CT2010", "buildingSharedWallsRatio"],
    key_on="feature.properties.CT2010",
    fill_color="YlGnBu",
    threshold_scale=myscale,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Building Shared Walls Ratio",
).add_to(_map)

myscale = (df['buildingShapeIndex'].quantile((0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1))).tolist()
choropleth_buildingShapeIndex = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Building Shape Index",
    data=df,
    columns=["CT2010", "buildingShapeIndex"],
    key_on="feature.properties.CT2010",
    fill_color="YlGnBu",
    threshold_scale=myscale,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Building Shape Index",
).add_to(_map)

myscale = (df['buildingOrientation'].quantile((0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1))).tolist()
choropleth_buildingOrientation = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Building Orientation",
    data=df,
    columns=["CT2010", "buildingOrientation"],
    key_on="feature.properties.CT2010",
    fill_color="YlGnBu",
    threshold_scale=myscale,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Building Orientation",
).add_to(_map)

myscale = (df['buildingPerimeterWall'].quantile((0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1))).tolist()
choropleth_buildingPerimeterWall = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Building Perimeter Wall",
    data=df,
    columns=["CT2010", "buildingPerimeterWall"],
    key_on="feature.properties.CT2010",
    fill_color="YlGnBu",
    threshold_scale=myscale,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Building Perimeter Wall",
).add_to(_map)

myscale = (df['buildingMeanInterbuildingDistance'].quantile((0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1))).tolist()
choropleth_buildingMeanInterbuildingDistance = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Mean Interbuilding Distance",
    data=df,
    columns=["CT2010", "buildingMeanInterbuildingDistance"],
    key_on="feature.properties.CT2010",
    fill_color="YlGnBu",
    threshold_scale=myscale,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Mean Interbuilding Distance",
).add_to(_map)

myscale = (df['buildingBuildingAdjacency'].quantile((0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1))).tolist()
choropleth_buildingBuildingAdjacency = folium.Choropleth(
    geo_data=df,
    name="Choropleth of Building Adjacency",
    data=df,
    columns=["CT2010", "buildingBuildingAdjacency"],
    key_on="feature.properties.CT2010",
    fill_color="YlGnBu",
    threshold_scale=myscale,
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Building Adjacency",
).add_to(_map)

style_function = lambda x: {'fillColor': '#ffffff', 
                            'color':'#000000', 
                            'fillOpacity': 0.1, 
                            'weight': 0.1}
highlight_function = lambda x: {'fillColor': '#000000', 
                                'color':'#000000', 
                                'fillOpacity': 0.50, 
                                'weight': 0.1}
tooltip = folium.features.GeoJson(
    df,
    style_function=style_function, 
    control=False,
    highlight_function=highlight_function, 
    tooltip=folium.features.GeoJsonTooltip(
        fields=['buildingHeight', 'buildingElongation', 'buildingFormFactor', 'buildingVolumeFacadeRatio', 'buildingSharedWallsRatio', 'buildingShapeIndex',
                'buildingOrientation', 'buildingPerimeterWall', 'buildingMeanInterbuildingDistance','buildingBuildingAdjacency'],
        aliases=['Building Height (Meters)', 'Building Elongation', 'Building Form Factor', 'Building Volume Facade Ratio', 'Building Shared Walls Ratio',
                 'Building Shape Index', 'Building Orientation', 'Building Perimeter Wall', 'Mean Interbuilding Distance', 'Building Adjacency'],
        style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;") 
    )
)
_map.add_child(tooltip)
_map.keep_in_front(tooltip)


folium.LayerControl().add_to(_map)
_map

In [ ]:
_map.save('cgis_map2.html')

## **Building Blocks**

In [ ]:
nta = gpd.read_file('https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/NYC_Neighborhood_Tabulation_Areas_2020/FeatureServer/0/query?where=1=1&outFields=*&outSR=4326&f=pgeojson')
nta = nta.to_crs('EPSG:26918')

williamsburg = nta[nta.NTAName == 'Williamsburg']

In [ ]:
buildings = gpd.read_file('data/buildings_bk.shp')
buildings = buildings.set_crs('EPSG:26918')
buildings = buildings.clip(williamsburg)

streets = gpd.read_file('data/streets_bk.shp')
streets = streets.set_crs('EPSG:26918')
streets = streets.clip(williamsburg)

tessellation = gpd.read_file('data/tessellation_bk.shp')
tessellation = tessellation.set_crs('EPSG:26918')
tessellation = tessellation.clip(williamsburg)

blocks = gpd.read_file('data/blocks_bk.shp')
blocks = blocks.set_crs('EPSG:26918')
blocks = blocks.clip(williamsburg)

In [ ]:
buildings_with_raster_values = gpd.read_file('data/brooklyn_buildings_with_raster_values.shp')
buildings_with_raster_values = buildings_with_raster_values.to_crs('EPSG:26918')
buildings_with_raster_values = buildings_with_raster_values.clip(williamsburg)

In [ ]:
cluster_labels = pd.read_csv('data/labels.csv')

buildings_with_cluster_labels = gpd.read_file('data/buildings_bk.shp')
buildings_with_cluster_labels = buildings_with_cluster_labels.set_crs('EPSG:26918')
buildings_with_cluster_labels['cluster'] = cluster_labels
buildings_with_cluster_labels = buildings_with_cluster_labels.clip(williamsburg)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(25,25))

buildings.plot(ax=ax)
streets.plot(color='black', ax=ax)

ax.set_axis_off();

plt.savefig('images/buildings.png', dpi=150)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(25,25))

tessellation.plot(edgecolor='red', ax=ax)
buildings.plot(color='grey', ax=ax)

ax.set_axis_off();

plt.savefig('images/tessellation.png', dpi=150)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(25,25))

blocks.plot(color=None, edgecolor='red', ax=ax)
buildings.plot(color='grey', ax=ax)

ax.set_axis_off();

plt.savefig('images/blocks.png', dpi=150)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(25,25))

buildings_with_cluster_labels.plot(column='cluster', ax=ax)
streets.plot(color='black', ax=ax)

ax.set_axis_off();

plt.savefig('images/building_clusters.png', dpi=150)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(50,20))

buildings_with_raster_values.plot(column='lst', cmap='Reds', ax=ax[0], legend=True)
streets.plot(color='black', ax=ax[0])
ax[0].set_axis_off();

buildings_with_raster_values.plot(column='ndvi', cmap='Greens', ax=ax[1], legend=True)
streets.plot(color='black', ax=ax[1])
ax[1].set_axis_off();

buildings_with_raster_values.plot(column='albedo', cmap='Greys', ax=ax[2], legend=True)
streets.plot(color='black', ax=ax[2])
ax[2].set_axis_off();

plt.tight_layout()
plt.savefig('images/raster_values_williamsburg.png', dpi=150)